In [2]:
import logging
import warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

from torch.utils.data import Dataset
from datasets import load_dataset, Audio
from typing import Tuple, Union, Optional
import glob
import json
from pathlib import Path
from tqdm import tqdm
import torch
import torchaudio
import soundfile as sf
import io
import os
from torch.utils.data import Dataset
from datasets import load_dataset, Audio
from typing import Tuple, Union, Optional
import glob
import json
from pathlib import Path
from tqdm import tqdm

In [3]:
import unicodedata
VI_ALPHABET = (
    "aàảãáạăằẳẵắặâầẩẫấậbcdđeèẻẽéẹêềểễếệghiìỉĩíị"
    "klmnoòỏõóọôồổỗốộơờởỡớợpqrstuùủũúụưừửữứựvxyỳỷỹýỵ"
    " '"
)

def normalize_transcript(text: str) -> str:
    """Lowercase, NFC normalize, remove special chars; preserve Vietnamese diacritics."""
    text = unicodedata.normalize("NFC", text.lower().strip())
    return "".join(c for c in text if c in VI_ALPHABET)

In [4]:
class ViMD(Dataset):
    """
    ViMD Dataset class thiết kế để thay thế LIBRISPEECH.
    Trả về tuple: (waveform, sample_rate, transcript)
    """

    def __init__(
        self,
        dataset_name: str = "ViMD",  
        split: str = "train",       
        target_sample_rate: Optional[int] = 16000,
        min_duration_sec: float = 0.3,
        max_silence_ratio: float = 0.95,
        filter_bad_samples: bool = True,
    ) -> None:
        self.dataset_name = dataset_name
        self.split = split
        self.target_sr = target_sample_rate
        self.min_duration_sec = min_duration_sec
        self.max_silence_ratio = max_silence_ratio
        self.filter_bad_samples = filter_bad_samples

        # Load dataset từ Hugging Face (giữ nguyên cấu trúc decode=False để xử lý bytes)
        print(f"Loading ViMD dataset split: {split}...")
#-------------------------------------------------------------KHÚC NÀY ĐÃ SỬA---------------------------------------------------        
        # self._dataset = load_dataset(dataset_name, split=split)
        if split == 'train':
            data_dir = r"D:\hf_datasets\hub\datasets--nguyendv02--ViMD_Dataset\snapshots\3a5b30157034e7eadd5c75fae1a820c6f9383398\data"
            train_files = sorted(glob.glob(os.path.join(data_dir, "train-*.parquet")))

            self._dataset = load_dataset(
                "parquet",
                data_files=train_files,
                split="train"
            )
        if split == 'test':
            data_dir = r"D:\hf_datasets\hub\datasets--nguyendv02--ViMD_Dataset\snapshots\3a5b30157034e7eadd5c75fae1a820c6f9383398\data"
            train_files = sorted(glob.glob(os.path.join(data_dir, "test-*.parquet")))

            self._dataset = load_dataset(
                "parquet",
                data_files=train_files,
                split="train"
            )
        if split == 'valid':
            data_dir = r"/kaggle/input/datasets/ilewanducki/vimd-valid/VIMD_VALID"
            train_files = sorted(glob.glob(os.path.join(data_dir, "valid-*.parquet")))

            self._dataset = load_dataset(
                "parquet",
                data_files=train_files,
                split="train"
            )
#-------------------------------------------------------------KHÚC NÀY ĐÃ SỬA--------------------------------------------------
        # Lọc các cột cần thiết để tối ưu bộ nhớ nếu cần
        self._dataset = self._dataset.select_columns(["audio", "text"])
        self._dataset = self._dataset.cast_column("audio", Audio(decode=False))

        self._valid_indices = list(range(len(self._dataset)))

        if self.filter_bad_samples:
            self._valid_indices = self._build_valid_indices()

    def _build_valid_indices(self):

        valid = []

        for idx in tqdm(range(len(self._dataset)), desc=f"Filtering {self.split}"):

            try:
                waveform, sr, transcript = self._load_raw(idx)

                if transcript is None:
                    continue

                if len(transcript.strip()) == 0:
                    continue

                duration = waveform.shape[-1] / sr

                if duration < self.min_duration_sec:
                    continue

                silence_ratio = (waveform.abs() < 1e-4).float().mean().item()

                if silence_ratio >= self.max_silence_ratio:
                    continue

                valid.append(idx)

            except Exception:
                continue

        print(f"Kept {len(valid)}/{len(self._dataset)} samples")

        return valid

    def _load_raw(self, idx: int) -> Tuple[torch.Tensor, int, str]:

        item = self._dataset[idx]

        transcript = item.get("text", "")

        audio_bytes = item["audio"]["bytes"]

        with io.BytesIO(audio_bytes) as f:
            array, sr = sf.read(f, dtype="float32")

        waveform = torch.from_numpy(array)

        if waveform.ndim > 1:
            waveform = waveform.mean(dim=-1)

        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)

        if self.target_sr and sr != self.target_sr:
            waveform = torchaudio.functional.resample(
                waveform,
                sr,
                self.target_sr
            )
            sr = self.target_sr

        return waveform, sr, transcript

    def __len__(self):

        return len(self._valid_indices)

    def __getitem__(self, n: int):

        idx = self._valid_indices[n]

        waveform, sr, transcript = self._load_raw(idx)

        transcript = normalize_transcript(transcript)

        return waveform, sr, transcript

In [ ]:
!pip -q install qwen_asr transformers datasets jiwer torch librosa soundfile accelerate 

In [ ]:
!pip -q uninstall transformers -y
!pip -q install transformers==4.57.6


In [5]:
import torch
import time
import gc
import os
import tempfile
import soundfile as sf
from tqdm import tqdm

from qwen_asr import Qwen3ASRModel
from jiwer import wer

MODEL_ID = "Qwen/Qwen3-ASR-0.6B"
DATASET_ID = "nguyendv02/ViMD_Dataset"
SPLIT = "valid"
SAMPLING_RATE = 16000
BATCH_SIZE = 16


def main():

    print(f"Đang tải model {MODEL_ID}...")
    model = Qwen3ASRModel.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        device_map="cuda:0",
        max_inference_batch_size=BATCH_SIZE,
        max_new_tokens=256,
    )

    print(f"Đang kết nối dataset...")
    dataset = ViMD(split="valid")

    predictions = []
    references = []

    batch_audio_paths = []
    batch_refs = []

    total_inference_time = 0
    count = 0

    print("Bắt đầu Batch Inference...")

    try:
        for item in tqdm(dataset, desc="Preparing batch", unit="sample"):

            waveform, sr, ground_truth = item

            if not ground_truth:
                continue

            ground_truth = str(ground_truth).strip()

            audio_array = waveform.squeeze().cpu().numpy()

            tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
            sf.write(tmp.name, audio_array, sr)

            batch_audio_paths.append(tmp.name)
            batch_refs.append(ground_truth)

            if len(batch_audio_paths) == BATCH_SIZE:

                start = time.time()

                with torch.no_grad():
                    results = model.transcribe(
                        audio=batch_audio_paths,
                        language=None
                    )

                end = time.time()

                total_inference_time += (end - start)

                for r, ref in zip(results, batch_refs):
                    predictions.append(normalize_transcript(r.text))
                    references.append(ref)

                count += len(batch_audio_paths)

                for p in batch_audio_paths:
                    if os.path.exists(p):
                        os.remove(p)

                batch_audio_paths = []
                batch_refs = []

                if count % 50 == 0:
                    print(
                        f"Đã xong {count} mẫu | Avg time/sample: {total_inference_time/count:.4f}s"
                    )

        # xử lý batch cuối
        if len(batch_audio_paths) > 0:

            start = time.time()

            with torch.no_grad():
                results = model.transcribe(
                    audio=batch_audio_paths,
                    language=None
                )

            end = time.time()

            total_inference_time += (end - start)

            for r, ref in zip(results, batch_refs):
                predictions.append(normalize_transcript(r.text))
                references.append(ref)

            count += len(batch_audio_paths)

            for p in batch_audio_paths:
                if os.path.exists(p):
                    os.remove(p)

    except Exception as e:
        print("Lỗi:", e)

    if count > 0:

        avg_wer = wer(references, predictions)
        avg_time = total_inference_time / count

        print("\n" + "=" * 50)
        print("KẾT QUẢ ĐÁNH GIÁ")
        print("=" * 50)
        print(f"Tổng số mẫu       : {count}")
        print(f"WER               : {avg_wer:.4f} ({avg_wer*100:.2f}%)")
        print(f"Time / sample     : {avg_time:.4f} s")
        print("=" * 50)

    del model
    gc.collect()
    torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

2026-03-15 02:46:15.696144: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773542775.717719     313 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773542775.724250     313 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773542775.741570     313 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773542775.741592     313 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773542775.741595     313 computation_placer.cc:177] computation placer alr

Đang tải model Qwen/Qwen3-ASR-0.6B...


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Đang kết nối dataset...
Loading ViMD dataset split: valid...


Filtering valid: 100%|██████████| 1900/1900 [00:34<00:00, 54.63it/s]


...

Đã xong 1200 mẫu | Avg time/sample: 1.1308s


Preparing batch:  84%|████████▍ | 1604/1900 [31:02<06:06,  1.24s/sample]

Đã xong 1600 mẫu | Avg time/sample: 1.1337s


Preparing batch: 100%|██████████| 1900/1900 [36:38<00:00,  1.16s/sample]



KẾT QUẢ ĐÁNH GIÁ
Tổng số mẫu       : 1900
WER               : 0.1505 (15.05%)
Time / sample     : 1.1348 s
